# Coloración de Grafos con Algoritmo Genético

## 1. Planteamiento del problema
Se busca resolver el problema de coloración de vértices para un grafo no dirigido $G = (V, E)$ con $19$ nodos y $46$ aristas. La meta es asignar un color a cada nodo cumpliendo dos condiciones:

1. Ningún par de nodos conectados por una arista $(u, v) \in E$ puede tener el mismo color
2. Usar la menor cantidad posible de colores distintos ($N_c$). De acuerdo con la instancia de referencia, el número cromático óptimo teórico es $N_c = 4$.

Se comparan $6$ configuraciones experimentales variando la probabilidad de mutación ($0.01, 0.05, 0.10$) y el lugar del punto de cruce (Fijo y Aleatorio).

In [1]:
import random
import time
import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 1000)

# Definición de la instancia: 19 vértices y 46 aristas no dirigidas
NUM_NODOS = 19
NC_OBJETIVO = 4  # Número cromático teórico de la diapositiva

ARISTAS = [
    (1, 2), (1, 4), (1, 16),
    (2, 3), (2, 4), (2, 5), (2, 6), (2, 7),
    (3, 7), (3, 11), (3, 19),
    (4, 5), (4, 8), (4, 9),
    (5, 6), (5, 9), (5, 10),
    (6, 7), (6, 11), (6, 14),
    (7, 11),
    (8, 9), (8, 12), (8, 16),
    (9, 10), (9, 12), (9, 13), (9, 15),
    (10, 13), (10, 14),
    (11, 14), (11, 19),
    (12, 15), (12, 16),
    (13, 14), (13, 15), (13, 17), (13, 18),
    (14, 18), (14, 19),
    (15, 16), (15, 17),
    (16, 17),
    (17, 18), (17, 19),
    (18, 19),
]

print(f"Instancia cargada: {NUM_NODOS} nodos y {len(ARISTAS)} aristas.")

Instancia cargada: 19 nodos y 46 aristas.


## 2. Parámetros y diseño del algoritmo genético

La siguiente tabla muestra las decisiones de los componentes para el algoritmo genético.

| Elemento | Configuración | Detalle |
| :--- | :--- | :--- |
| **Tipo de cromosoma** | Entero | Lista de $19$ enteros donde la posición $i$ guarda el color del nodo $i$ ($g_i \in \{1, \dots, 19\}$). |
| **Longitud** | $19$ | Corresponde al total de vértices del grafo ($|V| = 19$). |
| **Tamaño de población** | $6$ | Se mantienen $6$ soluciones activas en cada generación. |
| **Inicialización** | Aleatoria uniforme | Cada gen se sortea de forma independiente en $[1, 19]$. |
| **Infactibilidad** | Penalización del $50\%$ | Un cromosoma es infactible si tiene al menos una arista con ambos extremos del mismo color ($\text{conflictos} > 0$). |
| **Función fitness** | $\text{FO}$ (factible) / $0.5 \cdot \text{FO}$ (infactible) | Se maximiza $\text{FO} = 20 - \text{colores}$. Si hay conflictos, $\text{Fitness} = \text{FO} - 0.5 \cdot \text{FO}$. |
| **Selección** | Ruleta | Cada individuo tiene probabilidad $P_i = \text{Fitness}_i / \sum \text{Fitness}$ de ser elegido como padre. |
| **Probabilidad de cruce** | $0.90$ | Con probabilidad $0.90$ se cruzan los padres; de lo contrario, pasan sin cambios. |
| **Puntos de cruce** | $1$ punto | Un solo corte divide el cromosoma en dos bloques. |
| **Lugar de cruce** | **Fijo** (gen 9) / **Aleatorio** ($1$ a $18$) | En modo fijo se corta tras el gen $9$; en aleatorio se elige un punto entero en $[1, 18]$. |
| **Probabilidad de mutación** | $0.01$, $0.05$ y $0.10$ | Cada gen puede cambiar aleatoriamente a otro entero en $[1, 19]$ con probabilidad $p_m$. |
| **Reemplazo** | Sustitución de los $2$ peores | Los $2$ hijos reemplazan directamente a los $2$ individuos con menor fitness de la población. |
| **Criterio de paro** | $1{,}000$ iteraciones o $N_c = 4$ | Termina al llegar a $1{,}000$ iteraciones o al encontrar una coloración factible de $4$ colores. |

In [2]:
# Parámetros base del algoritmo
TAM_POBLACION = 6
PROB_CRUCE = 0.90
PUNTO_CRUCE_FIJO = 9
ITERACIONES_MAX = 1000


def contar_colores(cromosoma):
    # Regresa cuántos colores distintos utiliza un cromosoma
    return len(set(cromosoma))


def contar_conflictos(cromosoma):
    # Cuenta cuántas aristas conectan nodos adyacentes con el mismo color
    return sum(1 for u, v in ARISTAS if cromosoma[u - 1] == cromosoma[v - 1])


def evaluar(cromosoma):
    # Calcula colores, conflictos, función objetivo (FO) y fitness penalizado
    colores = contar_colores(cromosoma)
    conflictos = contar_conflictos(cromosoma)
    factible = (conflictos == 0)
    fo = (NUM_NODOS + 1) - colores
    fit = fo if factible else fo - 0.5 * fo
    return {
        "colores": colores,
        "conflictos": conflictos,
        "factible": factible,
        "fo": fo,
        "fitness": fit,
    }


def seleccionar_ruleta(poblacion, fitness_actual, rng):
    # Selecciona un padre mediante ruleta proporcional al fitness
    limite = rng.random() * sum(fitness_actual)
    acumulado = 0.0
    for individuo, fit in zip(poblacion, fitness_actual):
        acumulado += fit
        if limite <= acumulado:
            return individuo[:]
    return poblacion[-1][:]


def cruzar(padre1, padre2, lugar_cruce, rng):
    # Aplica cruce de 1 punto con probabilidad 0.9
    if rng.random() < PROB_CRUCE:
        corte = PUNTO_CRUCE_FIJO if lugar_cruce == "Fijo" else rng.randint(1, NUM_NODOS - 1)
        hijo1 = padre1[:corte] + padre2[corte:]
        hijo2 = padre2[:corte] + padre1[corte:]
        return hijo1, hijo2
    return padre1[:], padre2[:]


def mutar(cromosoma, prob_mutacion, rng):
    # Muta cada gen de forma independiente con probabilidad prob_mutacion
    nuevo = cromosoma[:]
    for i in range(NUM_NODOS):
        if rng.random() < prob_mutacion:
            nuevo[i] = rng.randint(1, NUM_NODOS)
    return nuevo

## 3. Ejemplo paso a paso (Excel)

Antes de correr las simulaciones y experimentos, se verifica la lógica de la iteración realizada en excel.

1. Con los 6 cromosomas iniciales (todos infactibles), la suma de fitness es $26.0$. Al sortear $r_1 = 0.238$ y $r_2 = 0.544$, la ruleta selecciona a la **Solución 2** ($P_1$) y a la **Solución 4** ($P_2$).
2. Como $r_{\text{cruce}} = 0.37 < 0.90$, se intercambian los genes 10 al 19 entre ambos padres.
3. Mutación ($p_m = 0.05$): En $H_1$ muta únicamente el **gen 4** y en $H_2$ también muta solo el **gen 4**.
4. Los dos hijos ($\text{Fitness} = 4.0$) sustituyen a las dos peores soluciones iniciales que en este caso son la **Solución 2** ($\text{Fitness} = 3.5$) y la **Solución 5** ($\text{Fitness} = 4.0$).

In [3]:
poblacion_inicial_ej = [
    [3, 15, 16, 18, 3, 9, 11, 11, 1, 14, 16, 9, 13, 14, 3, 9, 12, 1, 8],     # Sol 1
    [12, 16, 7, 9, 12, 5, 6, 10, 11, 2, 9, 6, 15, 1, 6, 1, 14, 3, 11],      # Sol 2
    [8, 2, 13, 4, 19, 10, 10, 12, 11, 4, 19, 12, 11, 13, 13, 2, 11, 8, 18], # Sol 3
    [16, 11, 1, 5, 17, 17, 16, 10, 6, 5, 6, 6, 17, 15, 6, 14, 12, 1, 18],   # Sol 4
    [4, 2, 16, 18, 10, 4, 16, 3, 16, 7, 14, 3, 13, 7, 11, 10, 9, 19, 9],    # Sol 5
    [18, 13, 4, 3, 4, 9, 4, 18, 19, 13, 17, 5, 1, 19, 19, 2, 7, 2, 16],     # Sol 6
]

# Tabla de evaluación y probabilidades de ruleta
tabla_ruleta = []
for idx, ind in enumerate(poblacion_inicial_ej, start=1):
    ev = evaluar(ind)
    tabla_ruleta.append({
        "Sol.": idx,
        "Colores": ev["colores"],
        "Conflictos": ev["conflictos"],
        "Factible": "Sí" if ev["factible"] else "No",
        "FO": ev["fo"],
        "Fitness": ev["fitness"],
    })

df_ruleta = pd.DataFrame(tabla_ruleta)
df_ruleta["Prob."] = df_ruleta["Fitness"] / df_ruleta["Fitness"].sum()
df_ruleta["Acumulada"] = df_ruleta["Prob."].cumsum()
df_ruleta["Lím. Inf."] = df_ruleta["Acumulada"].shift(1, fill_value=0.0)
df_ruleta["Lím. Sup."] = df_ruleta["Acumulada"]

print("Evaluación inicial y ruleta")
print(df_ruleta.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Selección, cruce en gen 9 y mutación en gen 4
p1, p2 = poblacion_inicial_ej[1], poblacion_inicial_ej[3]
h1 = p1[:9] + p2[9:]
h2 = p2[:9] + p1[9:]

h1[3] = 17
h2[3] = 19

print("\nHijos generados tras cruce y mutación")
for nombre, hijo in [("Hijo 1", h1), ("Hijo 2", h2)]:
    ev = evaluar(hijo)
    print(f"{nombre}: {hijo} -> Colores={ev['colores']}, Conflictos={ev['conflictos']}, FO={ev['fo']}, Fitness={ev['fitness']}")

# Reemplazo de los 2 peores
fitness_vals = df_ruleta["Fitness"].tolist()
orden_peores = sorted(range(TAM_POBLACION), key=lambda i: (fitness_vals[i], i))
nueva_pob = [ind[:] for ind in poblacion_inicial_ej]
nueva_pob[orden_peores[0]] = h1
nueva_pob[orden_peores[1]] = h2

print(f"\nReemplazo de soluciones {orden_peores[0]+1} y {orden_peores[1]+1}")
df_nueva = pd.DataFrame([
    {"Pos.": i + 1, **evaluar(ind)} for i, ind in enumerate(nueva_pob)
])
print(df_nueva.to_string(index=False))

Evaluación inicial y ruleta
 Sol.  Colores  Conflictos Factible  FO  Fitness  Prob.  Acumulada  Lím. Inf.  Lím. Sup.
    1       11           3       No   9   4.5000 0.1731     0.1731     0.0000     0.1731
    2       13           1       No   7   3.5000 0.1346     0.3077     0.1731     0.3077
    3        9           4       No  11   5.5000 0.2115     0.5192     0.3077     0.5192
    4       11           4       No   9   4.5000 0.1731     0.6923     0.5192     0.6923
    5       12           4       No   8   4.0000 0.1538     0.8462     0.6923     0.8462
    6       12           2       No   8   4.0000 0.1538     1.0000     0.8462     1.0000

Hijos generados tras cruce y mutación
Hijo 1: [12, 16, 7, 17, 12, 5, 6, 10, 11, 5, 6, 6, 17, 15, 6, 14, 12, 1, 18] -> Colores=12, Conflictos=2, FO=8, Fitness=4.0
Hijo 2: [16, 11, 1, 19, 17, 17, 16, 10, 6, 2, 9, 6, 15, 1, 6, 1, 14, 3, 11] -> Colores=12, Conflictos=4, FO=8, Fitness=4.0

Reemplazo de soluciones 2 y 5
 Pos.  colores  conflictos  fact

In [4]:
def ejecutar_corrida(prob_mutacion, lugar_cruce, semilla):
    # Se ejecuta el algoritmo genético con el criterio de paro
    rng = random.Random(semilla)
    t_inicio = time.perf_counter()

    poblacion = [[rng.randint(1, NUM_NODOS) for _ in range(NUM_NODOS)] for _ in range(TAM_POBLACION)]
    mejor_sol = None
    mejor_colores = None
    iter_hallazgo = 0

    def actualizar_mejor(cromosoma, iteracion_actual):
        nonlocal mejor_sol, mejor_colores, iter_hallazgo
        if contar_conflictos(cromosoma) == 0:
            c = contar_colores(cromosoma)
            if mejor_colores is None or c < mejor_colores:
                mejor_sol = cromosoma[:]
                mejor_colores = c
                iter_hallazgo = iteracion_actual

    for ind in poblacion:
        actualizar_mejor(ind, 0)

    iteracion = 0
    while iteracion < ITERACIONES_MAX and mejor_colores != NC_OBJETIVO:
        iteracion += 1
        fit_actual = [evaluar(ind)["fitness"] for ind in poblacion]

        padre1 = seleccionar_ruleta(poblacion, fit_actual, rng)
        padre2 = seleccionar_ruleta(poblacion, fit_actual, rng)

        hijo1, hijo2 = cruzar(padre1, padre2, lugar_cruce, rng)
        hijo1 = mutar(hijo1, prob_mutacion, rng)
        hijo2 = mutar(hijo2, prob_mutacion, rng)

        actualizar_mejor(hijo1, iteracion)
        actualizar_mejor(hijo2, iteracion)

        # Sustituir las 2 peores soluciones
        orden = sorted(range(TAM_POBLACION), key=lambda i: (fit_actual[i], i))
        poblacion[orden[0]] = hijo1
        poblacion[orden[1]] = hijo2

    tiempo_total = time.perf_counter() - t_inicio
    return {
        "mejor_solucion": mejor_sol,
        "colores": mejor_colores,
        "iter_hallazgo": iter_hallazgo,
        "iteraciones": iteracion,
        "tiempo": tiempo_total,
    }

## 4. Diseño experimental y resultados comparativos

Para evaluar el desempeño del algoritmo se probaron $6$ configuraciones cambiando tres probabilidades de mutación ($0.01, 0.05, 0.10$) y las dos estrategias de punto de cruce (Fijo y Aleatorio). Cada configuración se corrió $30$ veces con las mismas semillas ($1$ a $30$) para que la comparación sea justa.

In [5]:
PROBS_MUTACION = [0.01, 0.05, 0.10]
LUGARES_CRUCE = ["Fijo", "Aleatorio"]
NUM_CORRIDAS = 30

resultados_crudos = []
num_conf = 0

for pm in PROBS_MUTACION:
    for lugar in LUGARES_CRUCE:
        num_conf += 1
        for semilla in range(1, NUM_CORRIDAS + 1):
            res = ejecutar_corrida(pm, lugar, semilla)
            resultados_crudos.append({
                "Configuración": num_conf,
                "Prob. mutación": pm,
                "Lugar de cruce": lugar,
                "Semilla": semilla,
                "Factible": res["mejor_solucion"] is not None,
                "Colores": res["colores"],
                "Iter. hallazgo": res["iter_hallazgo"],
                "Tiempo (s)": res["tiempo"],
                "Solución": res["mejor_solucion"],
            })

df_experimentos = pd.DataFrame(resultados_crudos)

# Tabla resumen equivalente a la hoja 'Experimentos' de Excel
resumen = []
for (cfg, pm, lugar), grupo in df_experimentos.groupby(["Configuración", "Prob. mutación", "Lugar de cruce"]):
    factibles = grupo[grupo["Factible"]]["Colores"]
    resumen.append({
        "Conf.": cfg,
        "Prob. mutación": pm,
        "Lugar de cruce": lugar,
        "Corridas factibles": f"{int(grupo['Factible'].sum())}/30",
        "Mejor (colores)": int(factibles.min()),
        "Promedio": factibles.mean(),
        "Desv. est.": factibles.std(ddof=1),
        "Tiempo prom. (s)": grupo["Tiempo (s)"].mean(),
    })

df_resumen = pd.DataFrame(resumen)
print("- - RESULTADOS DE TODAS LAS CONFIGURACIONES - -")
print(df_resumen.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

- - RESULTADOS DE TODAS LAS CONFIGURACIONES - -
 Conf.  Prob. mutación Lugar de cruce Corridas factibles  Mejor (colores)  Promedio  Desv. est.  Tiempo prom. (s)
     1          0.0100           Fijo              19/30                6    7.5263      0.6967            0.0485
     2          0.0100      Aleatorio              15/30                6    7.7333      0.8837            0.0493
     3          0.0500           Fijo              23/30                5    6.8261      0.8869            0.0512
     4          0.0500      Aleatorio              21/30                5    6.5714      0.8106            0.0528
     5          0.1000           Fijo              25/30                5    6.3200      0.5568            0.0536
     6          0.1000      Aleatorio              25/30                5    6.5600      0.7681            0.0535


In [6]:
# buscamos la mejor solución obtenida
mejor_registro = df_experimentos[
    (df_experimentos["Configuración"] == 4) & (df_experimentos["Semilla"] == 8)
].iloc[0]

sol_ganadora = mejor_registro["Solución"]
eval_ganadora = evaluar(sol_ganadora)

print("- - MEJOR SOLUCIÓN OBTENIDA - -")
print(f"Configuración     : {mejor_registro['Configuración']} (pm={mejor_registro['Prob. mutación']}, cruce {mejor_registro['Lugar de cruce']})")
print(f"Semilla           : {mejor_registro['Semilla']}")
print(f"Iteración hallazgo: {mejor_registro['Iter. hallazgo']}")
print(f"Colores usados    : {eval_ganadora['colores']} -> {sorted(set(sol_ganadora))}")
print(f"Conflictos        : {eval_ganadora['conflictos']} (Factible: {eval_ganadora['factible']})")
print(f"FO / Fitness      : {eval_ganadora['fo']} / {eval_ganadora['fitness']}")
print(f"Cromosoma         : {sol_ganadora}")

df_nodos = pd.DataFrame({
    "Nodo": list(range(1, NUM_NODOS + 1)),
    "Color": sol_ganadora
})
print("\nAsignación por nodo:")
print(df_nodos.T.to_string(header=False))

- - MEJOR SOLUCIÓN OBTENIDA - -
Configuración     : 4 (pm=0.05, cruce Aleatorio)
Semilla           : 8
Iteración hallazgo: 171
Colores usados    : 5 -> [1, 8, 13, 15, 16]
Conflictos        : 0 (Factible: True)
FO / Fitness      : 15 / 15
Cromosoma         : [8, 13, 1, 1, 8, 1, 15, 8, 15, 16, 13, 13, 1, 15, 8, 15, 13, 16, 8]

Asignación por nodo:
Nodo   1   2  3  4  5  6   7  8   9  10  11  12  13  14  15  16  17  18  19
Color  8  13  1  1  8  1  15  8  15  16  13  13   1  15   8  15  13  16   8
